Import libraries

In [1]:
import pandas as pd
import numpy as np


Load cleaned files

In [2]:
matches = pd.read_csv('../DATA1/matches_cleaned.csv')
deliveries = pd.read_csv('../DATA1/deliveries_cleaned.csv')


QUICK SANITY CHECK

In [3]:
matches.head()
deliveries.head()


,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,extra_runs,total_runs,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,1,1,legbyes,0,None,None,None
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,0,0,None,0,None,None,None
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0,1,1,wides,0,None,None,None
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,SC Ganguly,0,0,0,None,0,None,None,None
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,P Kumar,SC Ganguly,0,0,0,None,0,None,None,None


Aggregate ball-by-ball → player-match level

In [4]:
player_match_runs = deliveries.groupby(
    ['match_id', 'batter']
)['batsman_runs'].sum().reset_index()


player_match_runs.rename(
    columns={'batter': 'batsman', 'batsman_runs': 'runs'},
    inplace=True
)
player_match_runs.head()


,match_id,batsman,runs
0,335982,AA Noffke,9
1,335982,B Akhil,0
2,335982,BB McCullum,158
3,335982,CL White,6
4,335982,DJ Hussey,12


FEATURE 1 – CAREER AVERAGE RUNS

In [5]:
career_avg = player_match_runs.groupby('batsman')['runs'].mean().reset_index()
career_avg.columns = ['batsman', 'career_avg_runs']

data = player_match_runs.merge(career_avg, on='batsman', how='left')   # merge feature


FEATURE 2 – RECENT FORM (LAST 5 MATCHES)

In [6]:
data['recent_form'] = data.groupby('batsman')['runs'] \
                           .rolling(window=5).mean() \
                           .reset_index(0, drop=True)

data['recent_form'].fillna(data['career_avg_runs'], inplace=True)


Feature 3: Venue Average

In [7]:
match_venue = matches[['id', 'venue']].copy()
match_venue.rename(columns={'id': 'match_id'}, inplace=True)

data = data.merge(match_venue, on='match_id', how='left')


In [8]:
data.columns


Index(['match_id', 'batsman', 'runs', 'career_avg_runs', 'recent_form',
       'venue'],
      dtype='object')

In [65]:
# Keep the correct venue column
data.rename(columns={'venue_y': 'venue'}, inplace=True)

# Remove duplicate column
data.drop(columns=['venue_x'], inplace=True)

# Verify
print(data.columns)


Index(['match_id', 'batsman', 'runs', 'career_avg_runs', 'recent_form',
       'venue_avg_runs', 'venue'],
      dtype='object')


VENUE AVERAGE (Player performance at venue)

In [10]:
# Player average runs at each venue
venue_avg = (
    data.groupby(['batsman', 'venue'])['runs']
    .mean()
    .reset_index()
)

venue_avg.columns = ['batsman', 'venue', 'venue_avg_runs']

# Merge back to main data
data = data.merge(venue_avg, on=['batsman', 'venue'], how='left')

print("Venue feature added ")
print(data.columns)


Venue feature added 
Index(['match_id', 'batsman', 'runs', 'career_avg_runs', 'recent_form',
       'venue', 'venue_avg_runs'],
      dtype='object')


VENUE AVERAGE FEATURE

In [13]:
# Extract opponent team per batsman per match from deliveries
opponent_map = (
    deliveries[['match_id', 'batter', 'bowling_team']]
    .drop_duplicates()
)

opponent_map.columns = ['match_id', 'batsman', 'opponent']


In [70]:
# Merge opponent into main data
data = data.merge(
    opponent_map,
    on=['match_id', 'batsman'],
    how='left'
)


In [14]:
data.columns


Index(['match_id', 'batsman', 'runs', 'career_avg_runs', 'recent_form',
       'venue', 'venue_avg_runs'],
      dtype='object')

In [72]:
# Player vs Team (opponent average)
opponent_avg = (
    data.groupby(['batsman', 'opponent'])['runs']
    .mean()
    .reset_index()
)

opponent_avg.columns = ['batsman', 'opponent', 'opponent_avg']


In [73]:
data = data.merge(
    opponent_avg,
    on=['batsman', 'opponent'],
    how='left'
)

data['opponent_avg'] = data['opponent_avg'].fillna(
    data['career_avg_runs']
)


In [74]:
data[['batsman', 'opponent', 'opponent_avg']].head()


,batsman,opponent,opponent_avg
0,AA Noffke,Kolkata Knight Riders,9.000000
1,B Akhil,Kolkata Knight Riders,0.000000
2,BB McCullum,Royal Challengers Bangalore,37.933333
3,CL White,Kolkata Knight Riders,15.833333
4,DJ Hussey,Royal Challengers Bangalore,25.625000


In [75]:
# Recent form - last 5 matches
data['recent_form_5'] = (
    data.groupby('batsman')['runs']
    .rolling(5, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)


In [76]:
# Matches played till current match
data['matches_played'] = data.groupby('batsman').cumcount() + 1


In [77]:
# Consistency: std dev of last 10 matches
data['run_consistency'] = (
    data.groupby('batsman')['runs']
    .rolling(10, min_periods=2)
    .std()
    .reset_index(level=0, drop=True)
)

data['run_consistency'] = data['run_consistency'].fillna(
    data['run_consistency'].mean()
)


In [78]:
# Rate of scoring 50+
data['fifty_rate'] = (
    data.groupby('batsman')['runs']
    .apply(lambda x: (x >= 50).expanding().mean())
    .reset_index(level=0, drop=True)
)


In [79]:
# Drop any remaining missing values
data = data.dropna()

# Save updated dataset
data.to_csv('../DATA1/final_dataset.csv', index=False)


In [80]:
data = data.sort_values(['batsman', 'match_id'])

data['next_match_runs'] = data.groupby('batsman')['runs'].shift(-1)

data.dropna(inplace=True)


In [ ]:
split_point = int(len(data) * 0.8)

train_data = data.iloc[:split_point]
test_data = data.iloc[split_point:]

X_train = train_data[['career_avg_runs', 'recent_form', 'venue_avg_runs']]
y_train = train_data['next_match_runs']

X_test = test_data[['career_avg_runs', 'recent_form', 'venue_avg_runs']]
y_test = test_data['next_match_runs']



In [82]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import joblib

feature_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

feature_pipeline.fit(X_train)

joblib.dump(feature_pipeline, '../models1/feature_pipeline.pkl')


['../models1/feature_pipeline.pkl']

In [83]:
data.to_csv('../DATA1/final_dataset.csv', index=False)
